# @decorator

## 1.함수 실행 시간 측정

In [ ]:
import time

def timer(func):
    def wrapper():
        start = time.time()
        func()
        end = time.time()
        print(f"실행 시간: {end - start:.3f}초")
    return wrapper

@timer
def slow_task():
    time.sleep(2)
    print("작업 완료!")

slow_task()


: 

## 2.로그 기록 남기기

In [ ]:
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"함수 {func.__name__} 시작!")
        result = func(*args, **kwargs)
        print(f"함수 {func.__name__} 끝!")
        return result
    return wrapper

@logger
def add(a, b):
    return a + b

add(3, 5)


## 3.가위바위보 게임 tool

- @tool 사용시 주의할 점
1. 독스트링은 매우 중요. 왜냐하면 AI는 이 설명을 읽고 도구를 선택하기 때문. 따라서 명확하고 구체적으로 작성해야 함.
2. 에러처리를 충실히 해야 함. AI가 예상치 못한 입력을 전달할 수 있기 때문. 입력과 출력이 자연어로 오는 상황은 대부분 잘 동작하지만 예상치 못한 에러가 종종 나타나게 됨. 따라서 최대한 모든 에러 상황을 고려하여 사용자 친화적인 메시지를 반환해야 함.

In [ ]:
# %pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

# .env 파일을 로드
load_dotenv()


openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key)

In [ ]:
import random
from langchain.tools import tool
from langchain_openai import ChatOpenAI


# 1. 가위바위보 게임을 위한 Tool 정의
@tool
def rps() -> str:
    """가위바위보 중 하나를 랜덤하게 선택 """
    return random.choice(["가위", "바위", "보"])

In [ ]:
# 2. Tool 바인딩된 LLM
llm = ChatOpenAI(api_key=openai_key, temperature=0.0).bind_tools([rps])
llm_for_chat = ChatOpenAI(temperature=0.7)  # 해설용 LLM
print(type(llm))

In [ ]:
# 3. 승부 판정
def judge(user_choice, computer_choice):
    """가위바위보 승패를 판정함."""
    user_choice = user_choice.strip()
    computer_choice = computer_choice.strip()
    if user_choice == computer_choice:
        return "무승부"
    elif (user_choice, computer_choice) in [
        ("가위", "보"),
        ("바위", "가위"),
        ("보", "바위"),
    ]:
        return "승리"
    else:
        return "패배"

In [ ]:
# 4. 게임 루프
print("가위바위보! (종료: q)")
while (user_input := input("\n가위/바위/보: ")) != "q":
    # LLM에게 tool 호출 요청
    ai_msg = llm.invoke(
        f"가위바위보 게임: 사용자가 {user_input}를 냈습니다. rps tool을 사용하세요."
    )
    # Tool 호출 확인 및 실행
    if ai_msg.tool_calls:
        print(type(rps))
        llm_choice = rps.invoke("")
        print(f"LLM이 선택한 도구: {llm_choice}")
        result = judge(user_input, llm_choice)

        print(f"승부: {result}")

        final = llm_for_chat.invoke(
            f"    "
            f"사용자: {user_input}, AI: {llm_choice}, 결과: 사용자의 {result}"
        )
        print(final)
        print(f"LLM 해설: {final.content}")
        print(f"게임 요약: 당신({user_input}) vs AI({llm_choice}) => {result}")
    else:
        print("Tool 호출 실패")